### **02 — Mesh cleaning and coronary skeletonisation**
This notebook prepares two inputs for coronary territory modelling:
1. A cleaned, triangulated myocardial surface.
2. A skeleton of the coronary artery segmentation.

| Input | Processing | Output |
|---|---|---|
| myocardium.stl | Mesh cleaning and triangulation | myocardium_cleaned.stl and myocardium_cleaned.vtp |
| label_16bit.nii | Coronary skeletonisation | label_skeleton.nii |


**Import libraries and select data folder**

PyMeshLab applies the mesh-cleaning filters. PyVista and VTK inspect the surface geometry and prepare the triangulated output. NiBabel loads and saves the coronary labels, and scikit-image performs skeletonisation. 

The mesh-cleaning stage processes patient subfolders containing `myocardium.stl`. Cases without this file are recorded as skipped.

In [1]:
import os
from collections import Counter
import numpy as np
import nibabel as nib
import pyvista as pv
import vtk
import pymeshlab
from skimage.morphology import skeletonize_3d
from pathlib import Path

plugin_dir = Path(pymeshlab.__file__).parent / "PlugIns"
plugin_names = ["libio_base.so", "libfilter_clean.so",  "libfilter_meshing.so",
    "libfilter_select.so", "libfilter_unsharp.so",]

for plugin_name in plugin_names:
    pymeshlab.load_plugin(str(plugin_dir / plugin_name))

base_dir = "/Users/ahthini/Documents/KCL/individual project/imageCAS_30_samples"
subject_folders = sorted([f for f in os.listdir(base_dir)
    if os.path.isdir(os.path.join(base_dir, f))])

results_summary = []

**Define the mesh-cleaning operations and mesh-quality checks**

The original cleaning sequence:
- reorients faces coherently
- inverts face orientation
- selects self-intersecting faces and expands that selection
- closes holes
- removes unreferenced vertices
- applies HC Laplacian smoothing
The resulting surface is saved as `myocardium_cleaned.stl`.

The below functions inspect checks like no. of vertices per face, boundary and non-manifold edges, connected components, zero-area and very small triangles. The analysis function triangulates the surface, computes consistent
normals, and saves `myocardium_cleaned.vtp`.

In [2]:
#apply meshlab cleaning filters to input mesh and save cleaned mesh to output path
def run_meshlab_cleaning(input_path, output_path):
    ms = pymeshlab.MeshSet()
    ms.load_new_mesh(input_path)
    ms.apply_filter("meshing_re_orient_faces_coherently")
    ms.apply_filter("meshing_invert_face_orientation")
    ms.apply_filter("compute_selection_by_self_intersections_per_face")
    ms.apply_filter("apply_selection_dilatation")
    ms.apply_filter("meshing_close_holes", maxholesize=100)
    ms.apply_filter("meshing_remove_unreferenced_vertices")
    ms.apply_filter("apply_coord_hc_laplacian_smoothing")
    ms.save_current_mesh(output_path, binary=False)
    print(f"[PyMeshLab] Saved cleaned mesh to: {output_path}")

#compute a histogram of face sizes in a mesh
def face_size_hist(poly: pv.PolyData) -> Counter:
    sizes = Counter()
    fa = poly.faces
    i = 0
    n = len(fa)
    while i < n:
        m = int(fa[i])
        sizes[m] += 1
        i += m + 1
    return sizes

#count the number of boundary and non-manifold edges in a mesh
def count_feature_edges(poly: pv.PolyData):
    f = vtk.vtkFeatureEdges()
    f.SetInputData(poly)
    f.BoundaryEdgesOn()
    f.NonManifoldEdgesOn()
    f.FeatureEdgesOff()
    f.ManifoldEdgesOff()
    f.Update()
    edges_all = pv.wrap(f.GetOutput())

    fb = vtk.vtkFeatureEdges()
    fb.SetInputData(poly)
    fb.BoundaryEdgesOn()
    fb.NonManifoldEdgesOff()
    fb.FeatureEdgesOff()
    fb.ManifoldEdgesOff()
    fb.Update()
    boundary = pv.wrap(fb.GetOutput())

    fn = vtk.vtkFeatureEdges()
    fn.SetInputData(poly)
    fn.BoundaryEdgesOff()
    fn.NonManifoldEdgesOn()
    fn.FeatureEdgesOff()
    fn.ManifoldEdgesOff()
    fn.Update()
    nonman = pv.wrap(fn.GetOutput())

    return edges_all, boundary, nonman

#number of connected components in a mesh
def num_components(poly: pv.PolyData) -> int:
    conn = vtk.vtkConnectivityFilter()
    conn.SetInputData(poly)
    conn.SetExtractionModeToAllRegions()
    conn.ColorRegionsOn()
    conn.Update()
    out = pv.wrap(conn.GetOutput())
    return int(out["RegionId"].max()) + 1 if "RegionId" in out.array_names else 1

#number of zero-area and tiny-area triangles in a mesh
def compute_zero_area_stats(poly: pv.PolyData, atol=1e-12):
    tri = poly.triangulate().clean()
    tri = tri.compute_cell_sizes(area=True)
    areas = np.asarray(tri["Area"])
    zero = int(np.sum(areas <= atol))
    tiny = int(np.sum((areas > atol) & (areas < 1e-6)))
    return zero, tiny, areas.min() if areas.size else None, areas.max() if areas.size else None

**Run mesh cleaning**

Apply cleaning and analysis functions to each patient folder.
The final summary reports the existing script's PASS, FAIL, or SKIP
verdict for each case.

In [3]:
#analyse a mesh and print a quality report, then save cleaned mesh to output directory
def analyse(stl_path: str, out_dir: str, subj: str):
    mesh = pv.read(stl_path)
    print("\n=== STL QUALITY REPORT ===")
    print(f"Path: {stl_path}")
    print(f"n_points: {mesh.n_points}")
    print(f"n_faces:  {mesh.n_faces}")
    print(f"is_all_triangles (PyVista): {mesh.is_all_triangles}")

    sizes = face_size_hist(mesh)
    print("\nFace-size breakdown (#points per face):")
    for k in sorted(sizes):
        tag = "Triangle" if k == 3 else "Non-triangle"
        print(f"  {k}-point faces ({tag}): {sizes[k]}")

    tri = mesh.triangulate().clean()
    print(f"  ↓ duplicate/merged points reduced: {mesh.n_points} -> {tri.n_points}")

    ncomp_raw = num_components(mesh)
    ncomp_tri = num_components(tri)
    print(f"Connected components (raw/tri-clean): {ncomp_raw} / {ncomp_tri}")

    _, boundary_raw, nonman_raw = count_feature_edges(mesh)
    _, boundary_tri, nonman_tri = count_feature_edges(tri)
    print(f"Boundary edges (tri-clean): {boundary_tri.n_cells}")
    print(f"Non-manifold edges (tri-clean): {nonman_tri.n_cells}")

    zero, tiny, _, _ = compute_zero_area_stats(mesh)
    zero_t, tiny_t, _, _ = compute_zero_area_stats(tri)
    print(f"Degenerate faces (tri-clean) -> zero: {zero_t}, tiny: {tiny_t}")

    #verdict checks
    issues = []
    if boundary_tri.n_cells > 0:
        issues.append("Has boundary edges")
    if nonman_tri.n_cells > 0:
        issues.append("Has non-manifold edges")
    if ncomp_tri != 1:
        issues.append(f"Has {ncomp_tri} disconnected components")
    if zero_t > 0 or tiny_t > 0:
        issues.append("Contains degenerate/tiny triangles")
    if tri.n_points < mesh.n_points:
        issues.append("Had duplicate/merged points")

    wt = (len(issues) == 0)
    print("\n--- VERDICT ---")
    print("PASS: Mesh ready for Voronoi" if wt else "FAIL: " + ", ".join(issues))

    #save cleaned surface
    norms = vtk.vtkPolyDataNormals()
    norms.SetInputData(tri)
    norms.SplittingOff()
    norms.ConsistencyOn()
    norms.AutoOrientNormalsOn()
    norms.ComputePointNormalsOff()
    norms.ComputeCellNormalsOn()
    norms.Update()
    tri_norm = pv.wrap(norms.GetOutput())
    norm_path = os.path.join(out_dir, "myocardium_cleaned.vtp")
    tri_norm.save(norm_path)
    print(f"Saved normals-consistent surface: {norm_path}")
    results_summary.append((subj, "PASS" if wt else "FAIL", ", ".join(issues) if issues else "None"))

if __name__ == "__main__":
    for subj in subject_folders:
        subj_dir = os.path.join(base_dir, subj)
        stl_path = os.path.join(subj_dir, "myocardium.stl")
        if not os.path.exists(stl_path):
            print(f"[SKIP] No myocardium.stl in {subj}")
            results_summary.append((subj, "SKIP", "No STL found"))
            continue
        out_stl = os.path.join(subj_dir, "myocardium_cleaned.stl")
        print(f"\n=== Processing {subj} ===")
        run_meshlab_cleaning(stl_path, out_stl)
        analyse(out_stl, subj_dir, subj)

    print("\n\n=== FINAL SUMMARY ===")
    print("{:<12} {:<6} {}".format("Folder", "Status", "Issues"))
    print("-" * 60)
    for folder, status, issues in results_summary:
        print("{:<12} {:<6} {}".format(folder, status, issues))


=== Processing 10064282 ===
[PyMeshLab] Saved cleaned mesh to: /Users/ahthini/Documents/KCL/individual project/imageCAS_30_samples/10064282/myocardium_cleaned.stl

=== STL QUALITY REPORT ===
Path: /Users/ahthini/Documents/KCL/individual project/imageCAS_30_samples/10064282/myocardium_cleaned.stl
n_points: 249170
n_faces:  498340
is_all_triangles (PyVista): True


/Users/ahthini/Documents/KCL/individual project/.venv/lib/python3.12/site-packages/pyvista/core/pointset.py:1365: PyVistaDeprecationWarning: The current behavior of `pv.PolyData.n_faces` has been deprecated.
                Use `pv.PolyData.n_cells` or `pv.PolyData.n_faces_strict` instead.
                See the documentation in '`pv.PolyData.n_faces` for more information.
  warnings.warn(



Face-size breakdown (#points per face):
  3-point faces (Triangle): 498340
  ↓ duplicate/merged points reduced: 249170 -> 249170
Connected components (raw/tri-clean): 1 / 1
Boundary edges (tri-clean): 0
Non-manifold edges (tri-clean): 0
Degenerate faces (tri-clean) -> zero: 0, tiny: 0

--- VERDICT ---
PASS: Mesh ready for Voronoi
Saved normals-consistent surface: /Users/ahthini/Documents/KCL/individual project/imageCAS_30_samples/10064282/myocardium_cleaned.vtp

=== Processing 10175956 ===
[PyMeshLab] Saved cleaned mesh to: /Users/ahthini/Documents/KCL/individual project/imageCAS_30_samples/10175956/myocardium_cleaned.stl

=== STL QUALITY REPORT ===
Path: /Users/ahthini/Documents/KCL/individual project/imageCAS_30_samples/10175956/myocardium_cleaned.stl
n_points: 232646
n_faces:  465292
is_all_triangles (PyVista): True

Face-size breakdown (#points per face):
  3-point faces (Triangle): 465292
  ↓ duplicate/merged points reduced: 232646 -> 232646
Connected components (raw/tri-clean): 

**Extract coronary skeletons**

Each coronary label is converted to a binary mask and reduced to a
one-voxel-wide skeleton using `skeletonize_3d`. The skeleton is saved as `label_skeleton.nii`, using the input label's affine and header.

In [4]:
subject_folders = sorted(os.listdir(base_dir))
for subject_folder in subject_folders:
    folder_path = os.path.join(base_dir, subject_folder)
    if not os.path.isdir(folder_path):
        continue

    label_path = os.path.join(folder_path, "label_16bit.nii")
    if not os.path.exists(label_path):
        print(f"✘ Missing label_16bit.nii in {subject_folder}")
        continue

    print(f"Processing skeleton: {subject_folder}")
    label_img = nib.load(label_path)
    label_data = label_img.get_fdata()
    binary_label = (label_data > 0).astype(np.uint8)
    skeleton = skeletonize_3d(binary_label).astype(np.uint8)

    #save skeleton
    skeleton_img = nib.Nifti1Image(skeleton, label_img.affine, label_img.header)
    skeleton_path = os.path.join(folder_path, "label_skeleton.nii")
    nib.save(skeleton_img, skeleton_path)
    print("Skeleton saved")

Processing skeleton: 10064282
Skeleton saved
Processing skeleton: 10175956
Skeleton saved
Processing skeleton: 10257303
Skeleton saved
Processing skeleton: 10423186
Skeleton saved
Processing skeleton: 10746739
Skeleton saved
Processing skeleton: 10814698
Skeleton saved
Processing skeleton: 10878112
Skeleton saved
Processing skeleton: 10974200
Skeleton saved
Processing skeleton: 11089463
Skeleton saved
Processing skeleton: 11109732
Skeleton saved
Processing skeleton: 11149298
Skeleton saved
Processing skeleton: 11699527
Skeleton saved
Processing skeleton: 11718071
Skeleton saved
Processing skeleton: 11934639
Skeleton saved
Processing skeleton: 11936127
Skeleton saved
Processing skeleton: 11963390
Skeleton saved
Processing skeleton: 11984870
Skeleton saved
Processing skeleton: 11996224
Skeleton saved
Processing skeleton: 12010440
Skeleton saved
Processing skeleton: 12017078
Skeleton saved
Processing skeleton: 12020071
Skeleton saved
Processing skeleton: 12060372
Skeleton saved
Processing